In [5]:
from pathlib import Path
import sys

In [6]:
project_root = Path.cwd()

if project_root.name == "notebooks":
    project_root = project_root.parent

sys.path.insert(0, str(project_root))
from src.database import get_engine
from sqlalchemy import text
import os
import getpass

os.environ["PGUSER"] = "sgurung"
os.environ["PGDATABASE"] = "citibike"
os.environ["PGPASSWORD"] = getpass.getpass("PostgreSQL password: ")

In [7]:
import openmeteo_requests

import pandas as pd
import requests_cache
from retry_requests import retry

In [8]:
cache_session = requests_cache.CachedSession('.cache', expire_after = -1)
retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
openmeteo = openmeteo_requests.Client(session = retry_session)

url = "https://archive-api.open-meteo.com/v1/archive"
params = {
	"latitude": 40.71,
	"longitude": -74,
	"start_date": "2025-01-01",
	"end_date": "2026-08-01",
	"hourly": ["temperature_2m", "dew_point_2m", "precipitation", "snowfall", "wind_speed_10m","weather_code"],
	"timezone": "America/New_York",
}

In [9]:
responses = openmeteo.weather_api(url, params = params)
response = responses[0]
hourly = response.Hourly()
hourly_temperature_2m = hourly.Variables(0).ValuesAsNumpy()
hourly_dew_point_2m = hourly.Variables(1).ValuesAsNumpy()
hourly_precipitation = hourly.Variables(2).ValuesAsNumpy()
hourly_snowfall = hourly.Variables(3).ValuesAsNumpy()
hourly_weather_code = hourly.Variables(4).ValuesAsNumpy()
hourly_wind_speed_10m = hourly.Variables(5).ValuesAsNumpy()

In [10]:
hourly_data = {
	"date": pd.date_range(
		start = pd.to_datetime(hourly.Time(), unit = "s", utc = True),
		end =  pd.to_datetime(hourly.TimeEnd(), unit = "s", utc = True),
		freq = pd.Timedelta(seconds = hourly.Interval()),
		inclusive = "left"
	).tz_convert(response.Timezone().decode())
}

hourly_data["temperature_2m"] = hourly_temperature_2m
hourly_data["dew_point_2m"] = hourly_dew_point_2m
hourly_data["precipitation"] = hourly_precipitation
hourly_data["snowfall"] = hourly_snowfall
hourly_data["weather_code"] = hourly_weather_code
hourly_data["wind_speed_10m"] = hourly_wind_speed_10m

In [11]:
hourly_dataframe = pd.DataFrame(data = hourly_data)
hourly_dataframe.head()

,date,temperature_2m,dew_point_2m,precipitation,snowfall,weather_code,wind_speed_10m
0,2024-12-31 23:00:00-05:00,8.05,7.00,3.4,0.0,17.468348,63.0
1,2025-01-01 00:00:00-05:00,9.05,8.30,0.9,0.0,12.425216,53.0
2,2025-01-01 01:00:00-05:00,8.65,8.35,0.0,0.0,4.965521,3.0
3,2025-01-01 02:00:00-05:00,8.85,8.65,0.0,0.0,8.287822,3.0
4,2025-01-01 03:00:00-05:00,8.75,8.45,0.0,0.0,9.299225,3.0


In [12]:
hourly_dataframe.info()

<class 'pandas.DataFrame'>
RangeIndex: 13872 entries, 0 to 13871
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype                          
---  ------          --------------  -----                          
 0   date            13872 non-null  datetime64[s, America/New_York]
 1   temperature_2m  13872 non-null  float32                        
 2   dew_point_2m    13872 non-null  float32                        
 3   precipitation   13872 non-null  float32                        
 4   snowfall        13872 non-null  float32                        
 5   weather_code    13872 non-null  float32                        
 6   wind_speed_10m  13872 non-null  float32                        
dtypes: datetime64[s, America/New_York](1), float32(6)
memory usage: 433.6 KB


In [13]:
engine = get_engine()
with engine.connect() as conn:
        date_range = pd.read_sql(
            text(f"""
                Select min(hour) as start_hour, max(hour) as end_hour
                from hourly_neighborhood_flow
                """),
                con=conn
        )
engine.dispose()
date_range

,start_hour,end_hour
0,2023-12-31 02:00:00,2026-07-31 23:00:00


In [14]:
date_range["start_hour"][0].strftime("%Y-%m-%d")

'2023-12-31'

In [15]:
engine = get_engine()
with engine.connect() as conn:
        weather_hist = pd.read_sql(
            text(f"""
                Select *
                from hourly_weather_history
                """),
                con=conn
        )

        weather_forecast = pd.read_sql(
            text(f"""
                Select *
                from hourly_weather_forecast
                """),
                con=conn
        )
engine.dispose()

display(weather_hist.head(),
        weather_forecast.head())

,hour,temperature,dew_point,precipitation,snowfall,weather_code,wind_speed
0,2023-12-30 23:00:00,2.30,0.00,0.0,0.0,0.0,7.729527
1,2023-12-31 00:00:00,1.20,-0.75,0.0,0.0,0.0,8.587338
2,2023-12-31 01:00:00,1.00,-1.00,0.0,0.0,0.0,7.729527
3,2023-12-31 02:00:00,0.60,-1.75,0.0,0.0,0.0,8.049845
4,2023-12-31 03:00:00,-0.45,-2.75,0.0,0.0,0.0,7.968939


,hour,temperature,dew_point,precipitation,snowfall,weather_code,wind_speed,forecast_generated_at
0,2026-09-07 16:00:00,26.793499,8.722998,0.0,0.0,0.0,12.605142,2026-09-07 21:06:52.536223+00:00
1,2026-09-07 17:00:00,26.893500,8.340814,0.0,0.0,0.0,13.246826,2026-09-07 21:06:52.536223+00:00
2,2026-09-07 18:00:00,26.043499,7.603480,0.0,0.0,0.0,9.449572,2026-09-07 21:06:52.536223+00:00
3,2026-09-07 19:00:00,24.043499,8.452909,0.0,0.0,0.0,8.209263,2026-09-07 21:06:52.536223+00:00
4,2026-09-07 20:00:00,22.143500,8.272043,0.0,0.0,0.0,7.235910,2026-09-07 21:06:52.536223+00:00
